In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw"
spark.sql(f"LIST '{VOLUME_PATH}'").display()

In [0]:
df = spark.sql("FROM marathos.bronze.raw_supply_chain")
df.display()

In [0]:
df.count()

In [0]:
len(df.columns)

In [0]:
df.printSchema()

### Schema insigts 

* Athlete year of birht: was inferred as a double but should be integer. Probably caused by null values. 
* Athelte average speed: was inferred as a string but should be double. Now it suggests non-numeric values are present and that will have to be investigated in Silver.
* Athelte performance: is a string and that might need to be corrected to a timestamp but it is possible that might not work and will have to be a double and calculated.

This will be done in silver EDA/Cleaning

In [0]:
df.describe().display()

* athlete year of birth seems to have som input error considering there is someone born 1193 and somone born 2021.
* Athlete performance contains times that cover days wich should be dropped in silver.
* There are actually data from 1798 wich is very cool! 
* Mean/stddv are meaningless for club and distance sins they are strings.


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count the number of null values in each column
null_counts = df.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

# Convert the result to a dictionary so that we can loop through it
null_counts = null_counts.collect()[0].asDict()

# Only keep volumes with null values
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

### Null count reflection 

* Athlete club is high but this is not unlikley due to club membership probably being optional. 
* Athlete performance country and gender are so smal they will probably be droped in silver
* Athlete average speed might is something that we can calculate depending on the other data for that person, Meaning if we have there time and distnace, will otherwise be dropped.

In [0]:
# how many unique events are their?

df.select("Event name").distinct().count()

In [0]:
import plotly.express as px

age_dist = (
    df.groupBy("Athlete age category")
    .count()
    .filter(col("Athlete age category").isNotNull())
    .orderBy("Athlete age category")
    .toPandas()
)

fig = px.histogram(
    age_dist,
    x = "Athlete age category",
    y = "count",
    title = "Age distribution of athletes",
    labels = {"Athlete age category": "Age category", "count": "Number of athletes"},
)

fig.show()

In [0]:
country_dist = (
    df.groupBy("Athlete country")
    .count()
    .filter(col("Athlete country").isNotNull())
    .toPandas()
    .sort_values("count", ascending=False)
    .head(15)
)

fig = px.bar(
    country_dist,
    x = "Athlete country",
    y = "count",
    title = "Top 15 most represented countries",
    labels = {"Athlete country": "Country", "count": "Number of athletes"},
    text = "count",
)

fig.show()

# To do in silver EDA 

* snake_case for all column names
* change datatyp for Athlete year of birth, Athlete average speed and maybe Athlete performance
* Check out and drop null values. 
* make sure dates are correct. 
* some average speeds do not make sense for example 10000. Change or drop these. 